![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Day 44 -- Case Study: The Glitch Hunter (Solution)

The task of this morning's case study, **"The Glitch Hunter"**, from the Romanian AI olympiad's practice platform
MLCompete: competition *Algorithms to Neuron, Practice Round 1* (the **ONIA Winter Warmup 2025**). The same notebook, solved the way the Day 44 slides solve it, one step at a time, each step scored on validation images.

**Sources and credits.** The task, its statement and its data: [MLCompete](https://platform.olimpiada-ai.ro)
(all rights reserved by the platform), told here in our words with short quotes. The official editorial:
A.-D. Ardelean, *Editorial: The Glitch Hunter*, MLCompete, 13 December 2025 (in Romanian). A contestant's solution:
S. Asandei, [roai-solved](https://github.com/stefanasandei/roai-solved) (`contests/mlcompete/contests/winter_2025`,
MIT licence). The code of this notebook is the course's own, written for the Day 44 slides.

**How to run it.** The data is not copied here. Make a free account on the platform, open the task, download its
data, and put the folders `train/images`, `train/masks` and `test/images` in a folder `glitch` next to this notebook
(in Colab: upload them, or mount your Drive and change `DATA_DIR`). Choose a **GPU** runtime.


## The task, in our words

**The story.** You are an "Agent". A simulation is degrading: an AI model, *Project G3P1T*, paints glitches on its
images, "deliberate insertions of chaos". Your mission: find them, pixel by pixel.

**A glitch** is one of three things: a block of random digital **noise**, a rectangle with **inverted colours**, or a
compact block of **colour** that does not belong in the image.

**One data point** is one image, `train/images/img_001.png`, an array of shape (512, 512, 3). **The answer** is its
mask, `train/masks/mask_001.png`, shape (512, 512): 255 on a glitch pixel, 0 on a clean one.

**The files.** 200 training images with their masks; 200 test images without.

**What you hand in.** A ZIP of 200 PNG masks, named like the images (`img_001.png`), values 0 and 255.

**The score.** IoU (intersection over union) of your masks against the hidden ones: the pixels both mark, divided by
the pixels either marks. Higher is better. The statement does not say whether IoU is taken per image or over all
pixels; here we take the mean of the per-image IoU. (The page also shows a scoring snippet for a regression task,
`r2_score` and `pearsonr`: a leftover from another task, not this one.)


## Setup

In [ ]:
import os
import zipfile
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

DATA_DIR = "glitch"       # holds train/images, train/masks, test/images (downloaded from the platform)
names = sorted(os.listdir(os.path.join(DATA_DIR, "train", "images")))
test_names = sorted(os.listdir(os.path.join(DATA_DIR, "test", "images")))
print(len(names), "training images,", len(test_names), "test images")

## Load

In [ ]:
images = []
masks = []
for name in names:
    images.append(np.array(Image.open(os.path.join(DATA_DIR, "train", "images", name)).convert("RGB")))
    mask_name = name.replace("img_", "mask_")
    mask = np.array(Image.open(os.path.join(DATA_DIR, "train", "masks", mask_name)).convert("L"))
    masks.append(mask > 127)                  # True on a glitch pixel
images = np.stack(images)
masks = np.stack(masks)
print(images.shape, masks.shape)

## Look: one image and its mask

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].imshow(images[2])
axes[0].set_title(names[2])
shown = axes[1].imshow(masks[2], cmap="gray", vmin=0, vmax=1)
axes[1].set_title("its mask")
plt.colorbar(shown, ax=axes[1])
plt.show()
print("glitch pixels:", round(100 * masks.mean(), 1), "% of all training pixels")

## The score: IoU, and two answers with no model

In [ ]:
def iou(true_mask, pred_mask):
    both = np.logical_and(true_mask, pred_mask).sum()
    either = np.logical_or(true_mask, pred_mask).sum()
    if either == 0:
        return 1.0
    return both / either


def mean_iou(true_masks, pred_masks):
    scores = []
    for t, p in zip(true_masks, pred_masks):
        scores.append(iou(t, p))
    return np.mean(scores)


print("all clean:", mean_iou(masks, np.zeros_like(masks)))
print("all glitch:", round(mean_iou(masks, np.ones_like(masks)), 3))

## The file: a ZIP of PNG masks

In [ ]:
def write_zip(pred_masks, path="submission.zip"):
    """One PNG per test image, the image's own name, values 0 and 255."""
    os.makedirs("submission", exist_ok=True)
    with zipfile.ZipFile(path, "w") as archive:
        for name, mask in zip(test_names, pred_masks):
            Image.fromarray(mask.astype(np.uint8) * 255).save(os.path.join("submission", name))
            archive.write(os.path.join("submission", name), arcname=name)
    print(path, "holds", len(pred_masks), "masks")

In [ ]:
# A first valid file: every pixel clean. It scores 0; any model must beat it.
write_zip(np.zeros((len(test_names), 512, 512), dtype=bool))

## Split: keep 40 images to score on

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.models import resnet34, ResNet34_Weights

torch.manual_seed(42)
np.random.seed(42)
order = np.random.permutation(len(names))
val_ids = order[:40]                  # 40 images to score on, never trained on
train_ids = order[40:]
device = "cuda" if torch.cuda.is_available() else "cpu"
print(len(train_ids), "training images,", len(val_ids), "validation images, on", device)

## Batches: the image and its mask turned and mirrored together

In [ ]:
MEAN = np.array([0.485, 0.456, 0.406])
STD = np.array([0.229, 0.224, 0.225])
SIZE = 256                            # train at 256: resizing the true masks costs under 0.01 of IoU


class GlitchData(Dataset):
    def __init__(self, images, masks, augment):
        self.images = images
        self.masks = masks
        self.augment = augment

    def __len__(self):
        return len(self.images)

    def __getitem__(self, i):
        image = np.array(Image.fromarray(self.images[i]).resize((SIZE, SIZE), Image.BILINEAR))
        mask = np.array(Image.fromarray(self.masks[i].astype(np.uint8) * 255).resize((SIZE, SIZE))) > 127
        if self.augment:
            k = np.random.randint(4)              # the same quarter turns for the image and its mask
            image = np.rot90(image, k)
            mask = np.rot90(mask, k)
            if np.random.rand() < 0.5:            # the same mirror for both
                image = image[:, ::-1]
                mask = mask[:, ::-1]
        x = (image / 255 - MEAN) / STD
        x = torch.tensor(x.transpose(2, 0, 1).copy(), dtype=torch.float32)
        y = torch.tensor(mask.copy(), dtype=torch.float32).unsqueeze(0)
        return x, y


train_loader = DataLoader(GlitchData(images[train_ids], masks[train_ids], True), batch_size=8, shuffle=True)
val_loader = DataLoader(GlitchData(images[val_ids], masks[val_ids], False), batch_size=8)
xb, yb = next(iter(train_loader))
print(xb.shape, yb.shape)

## The model: a U-Net with a pretrained ResNet-34 encoder

In [ ]:
def conv_block(c_in, c_out):
    return nn.Sequential(nn.Conv2d(c_in, c_out, 3, padding=1), nn.BatchNorm2d(c_out), nn.ReLU(),
                         nn.Conv2d(c_out, c_out, 3, padding=1), nn.BatchNorm2d(c_out), nn.ReLU())


class UNet(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        resnet = resnet34(weights=ResNet34_Weights.DEFAULT)
        self.enc0 = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu)
        self.enc1 = nn.Sequential(resnet.maxpool, resnet.layer1)
        self.enc2 = resnet.layer2
        self.enc3 = resnet.layer3
        self.enc4 = resnet.layer4
        self.encoder = [self.enc0, self.enc1, self.enc2, self.enc3, self.enc4]
        self.up3 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.dec3 = conv_block(512, 256)
        self.up2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec2 = conv_block(256, 128)
        self.up1 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec1 = conv_block(128, 64)
        self.up0 = nn.ConvTranspose2d(64, 64, 2, stride=2)
        self.dec0 = conv_block(128, 64)
        self.up_out = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.head = nn.Conv2d(32, n_classes, 1)

    def forward(self, x):
        e0 = self.enc0(x)
        e1 = self.enc1(e0)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        e4 = self.enc4(e3)
        d3 = self.dec3(torch.cat([self.up3(e4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        d0 = self.dec0(torch.cat([self.up0(d1), e0], dim=1))
        return self.head(self.up_out(d0))


model = UNet(1).to(device)
print(model(xb.to(device)).shape)

## Training: frozen, then fine-tuned

In [ ]:
def train(model, epochs, lr):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.BCEWithLogitsLoss()
    for epoch in range(epochs):
        model.train()
        total = 0
        for x, y in train_loader:
            x = x.to(device)
            y = y.to(device)
            optimizer.zero_grad()
            loss = loss_fn(model(x), y)
            loss.backward()
            optimizer.step()
            total = total + loss.item()
        print("epoch", epoch, "loss", round(total / len(train_loader), 4))


# Stage 1: the encoder frozen
for part in model.encoder:
    for p in part.parameters():
        p.requires_grad = False
train(model, 5, 1e-3)

# Stage 2: everything learns, ten times more gently
for part in model.encoder:
    for p in part.parameters():
        p.requires_grad = True
train(model, 20, 1e-4)

## Validation: the mirror image too, and the threshold chosen on validation

In [ ]:
def probabilities(model, images, mirror):
    """The glitch probability of every pixel at 512 x 512; with mirror, averaged with the mirrored image's."""
    model.eval()
    no_masks = np.zeros(images.shape[:3], dtype=bool)
    loader = DataLoader(GlitchData(images, no_masks, False), batch_size=8)
    out = []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            p = torch.sigmoid(model(x))
            if mirror:
                p = (p + torch.sigmoid(model(x.flip(3))).flip(3)) / 2
            p = F.interpolate(p, size=(512, 512), mode="bilinear", align_corners=False)
            out.append(p[:, 0].cpu().numpy())
    return np.concatenate(out)


val_probs = probabilities(model, images[val_ids], mirror=True)
best_t = 0.5
best_score = 0
for t in [0.3, 0.4, 0.5, 0.6, 0.7]:
    s = mean_iou(masks[val_ids], val_probs > t)
    print("threshold", t, "IoU", round(s, 4))
    if s > best_score:
        best_score = s
        best_t = t
print("best threshold", best_t)

## The file

In [ ]:
test_images = []
for name in test_names:
    test_images.append(np.array(Image.open(os.path.join(DATA_DIR, "test", "images", name)).convert("RGB")))
test_images = np.stack(test_images)
write_zip(probabilities(model, test_images, mirror=True) > best_t)

## What each step gave on the slides

Measured on 40 validation images of the 200 (the slides' script), one change at a time:

| Step | Mean IoU |
|---|---|
| all glitch, no model | 0.065 |
| a U-Net from zero, threshold 0.5 | 0.464 |
| the image and its mask augmented together | 0.471 |
| a pretrained ResNet-34 encoder, frozen then fine-tuned | 0.894 |
| the mirror image at test time | 0.897 |
| the threshold chosen on validation (0.7) | 0.909 |
| every piece snapped to its box | 0.861 |

This notebook trains with a plain BCE loss and its own split, so its numbers will differ a little.

Made By **Sattam Altwaim**, for the Day 44 case study; the task belongs to MLCompete.